# NB5 — Apache Iceberg & the Catalog as Control Plane

**Stack:** `pyiceberg` + a local SQLite catalog. No JVM, no server, no cloud.
Maps to slide §4 (Apache Iceberg) + §12 (Catalog = Control Plane) + deliverable bullet 5.

NB1–NB4 used Delta. Here you meet the *other* open table format — and more
importantly, the thing both formats now agree is the centre of the
architecture: **the catalog**.

> **Catalog responsibilities.** This notebook uses a local SQLite catalog
> for namespaces, table registration and metadata updates. It demonstrates
> client-side scan planning, not remote planning or a security boundary.

> Production equivalent: `SqlCatalog(sqlite)` ↔ `load_catalog(type="rest")`
> against a compatible REST catalog. Production catalogs require separate
> configuration, authentication and checking which APIs/features they support.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)

import datetime as dtm

import pyarrow as pa

from lakehouse import catalog, count_files, du, human, namespace, reset_catalog

CAT = "nb5"          # own catalog dir: NB6/NB8/`make smoke` cannot disturb it
reset_catalog(CAT)   # idempotent rerun
cat = catalog(CAT)
ns = namespace(cat, "lake")
print(f"Catalog: {type(cat).__name__}   namespaces: {cat.list_namespaces()}")

ModuleNotFoundError: No module named '_setup'

## 1. Create a table *through the catalog*

Note what you do **not** do: you never pick a path. The catalog owns the
layout. That indirection is exactly what lets the catalog later vend
credentials, enforce row filters, and plan scans on your behalf.

`nullable=False` is not cosmetic — Iceberg tracks *required* vs *optional*
per field, and a required field can never be silently dropped.

In [ ]:
SCHEMA = pa.schema([
    pa.field("event_id",   pa.int64(),           nullable=False),
    pa.field("ts",         pa.timestamp("us"),   nullable=False),
    pa.field("model",      pa.string()),
    pa.field("latency_ms", pa.int64()),
    pa.field("cost_usd",   pa.float64()),
])

tbl = cat.create_table(f"{ns}.llm_events", schema=SCHEMA)
print(f"Created {tbl.name()}")
print(f"  location:  {tbl.location()}")
print(f"  metadata:  {tbl.metadata_location.rsplit('/', 1)[-1]}")
print(f"  format-v{tbl.format_version}")

Created ('lake', 'llm_events')
  location:  file://C:\Users\Admin\Desktop\ai-app-lab\L18-D18-work\K4-Track02-Day18-LeDucTung-03005-Lakehouse-Lab\_lakehouse\iceberg\nb5\warehouse/lake/llm_events
  metadata:  00000-d5ff5115-7c49-4d80-8d78-69b246c922ad.metadata.json
  format-v2


### 📝 Nhận xét — Catalog là control plane

- `cat.create_table("lake.llm_events", ...)` **không nhận đường dẫn**: catalog tự quyết định `location` (`warehouse/lake/llm_events`) và ghi file metadata đầu tiên `00000-….metadata.json`. Người dùng chỉ biết **tên bảng**.
- Catalog giữ **con trỏ tới metadata.json hiện tại** của mỗi bảng. Mỗi commit tạo metadata.json mới, rồi catalog đổi con trỏ một cách nguyên tử — đây là điểm commit ACID của Iceberg (tương đương việc ghi file JSON kế tiếp vào `_delta_log/` của Delta ở NB1).
- Vì mọi truy cập đi qua catalog, ở production catalog (REST: Polaris, Unity, Glue…) còn có thể cấp credential, kiểm soát quyền và lập kế hoạch scan thay client. Lab chỉ dùng **SQLite cục bộ** nên minh hoạ phần đăng ký bảng/namespace và cập nhật metadata, **không** minh hoạ bảo mật hay server-side planning.
- `format-v2`, `event_id`/`ts` khai báo `nullable=False` → Iceberg lưu trường *required*, không thể bị bỏ âm thầm.


## 2. Hidden partitioning — the feature that killed Hive

In Hive you partitioned by a **derived column** (`dt=2026-08-05`) that the
user had to know about and filter on by hand. Forget it, and you full-scan
the table. Every data team has that outage story.

Iceberg stores the *transform* (`day(ts)`) in metadata instead. You filter on
`ts` — the real column — and the engine derives the partition itself.

In [ ]:
from pyiceberg.transforms import DayTransform  # noqa: E402

with tbl.update_spec() as spec:
    spec.add_field("ts", DayTransform(), "ts_day")

tbl = cat.load_table(f"{ns}.llm_events")  # refresh after a metadata change
print(f"Partition spec: {tbl.spec()}")
print("\nNote: 'ts_day' is NOT a column you insert. It is derived from ts.")

Partition spec: [
  1000: ts_day: day(2)
]

Note: 'ts_day' is NOT a column you insert. It is derived from ts.


### 📝 Nhận xét — Hidden partitioning

- Spec `1000: ts_day: day(2)`: partition field `ts_day` (id 1000) = **phép biến đổi `day`** áp lên field-id **2** (`ts`). Metadata lưu *cách tính* partition, không lưu một cột phái sinh.
- `ts_day` **không phải cột** của bảng: ta không chèn nó, Iceberg tự tính khi ghi và đặt file vào `data/ts_day=2026-08-05/…`.
- Khác Hive: Hive bắt người dùng tạo cột `dt` và **tự nhớ** lọc `WHERE dt = ...`; người dùng Iceberg lọc trên cột thật `ts`, engine tự suy ra partition.


## 3. Append 10 daily batches

Each `append()` is one atomic commit → one snapshot. This is the same
ACID guarantee Delta gives you, expressed through a different metadata tree.

In [ ]:
MODELS = ["claude-haiku-4-5", "claude-sonnet-4-6", "claude-opus-4-7"]
COST_PER_CALL = {"claude-haiku-4-5": 0.0004, "claude-sonnet-4-6": 0.003, "claude-opus-4-7": 0.015}
ROWS_PER_DAY = 500
N_DAYS = 10


def day_batch(day: int) -> pa.Table:
    """One day of synthetic inference traffic."""
    models = [MODELS[i % 3] for i in range(ROWS_PER_DAY)]
    return pa.table({
        "event_id":   [day * ROWS_PER_DAY + i for i in range(ROWS_PER_DAY)],
        "ts":         [dtm.datetime(2026, 8, day, 3, i % 60) for i in range(ROWS_PER_DAY)],
        "model":      models,
        "latency_ms": [200 + (i * 7) % 3000 for i in range(ROWS_PER_DAY)],
        "cost_usd":   [COST_PER_CALL[m] for m in models],
    }, schema=SCHEMA)


for day in range(1, N_DAYS + 1):
    tbl.append(day_batch(day))

tbl = cat.load_table(f"{ns}.llm_events")
print(f"Rows:      {tbl.scan().to_arrow().num_rows:,}")
print(f"Snapshots: {len(tbl.snapshots())}   (one per commit)")
print(f"Data files: {tbl.inspect.files().num_rows}")

Rows:      5,000
Snapshots: 10   (one per commit)
Data files: 10


### 📝 Nhận xét — 10 append = 10 snapshot

- Mỗi `append()` là **một commit nguyên tử → một snapshot** (5.000 dòng, 10 snapshot, 10 data file — mỗi ngày 1 file 500 dòng trong partition `ts_day` của ngày đó).
- Snapshot là đơn vị của time travel và rollback trong Iceberg, tương đương version trong Delta (NB3).


## 4. Scan planning — count the files selected by the client

`plan_files()` is the planning step itself: given a filter, which data files
must actually be read? This lab plans the scan locally through PyIceberg;
it does not demonstrate server-side planning.

Watch: we filter on **`ts`**, never on `ts_day`.

In [ ]:
scan_all = tbl.scan()
scan_one_day = tbl.scan(row_filter="ts >= '2026-08-05T00:00:00' and ts < '2026-08-06T00:00:00'")

files_all = len(list(scan_all.plan_files()))
files_one = len(list(scan_one_day.plan_files()))

print(f"Files to read, no filter:    {files_all}")
print(f"Files to read, one-day filter: {files_one}")
print(f"→ Pruning ratio: {files_all / max(files_one, 1):.0f}×   (target ≥ 5×)")
print(f"  rows returned: {scan_one_day.to_arrow().num_rows:,}")

PRUNE_RATIO = files_all / max(files_one, 1)
assert PRUNE_RATIO >= 5, f"expected ≥5x pruning, got {PRUNE_RATIO:.1f}x"

Files to read, no filter:    10
Files to read, one-day filter: 1
→ Pruning ratio: 10×   (target ≥ 5×)
  rows returned: 500


### 📝 Nhận xét — Pruning qua scan planning

- `plan_files()` là bước **lập kế hoạch**: từ filter, quyết định data file nào phải đọc — chưa đọc dữ liệu nào.
- Không filter: **10 file**. Filter `ts >= '2026-08-05' AND ts < '2026-08-06'`: **1 file** ⇒ **pruning 10×** (≥ 5× ✅), trả đúng 500 dòng của ngày 05/08.
- Filter đặt trên **`ts`**, không phải `ts_day`: planner áp transform `day()` lên khoảng `ts` để suy ra chỉ cần `ts_day = 2026-08-05`, rồi so với giá trị partition lưu trong manifest để loại 9 file còn lại mà không mở chúng.


### The Hive comparison, made concrete

A Hive-style user who forgets the partition predicate reads **all** files.
The Iceberg user cannot make that mistake — there is no partition column to
forget. Same query intent, an order of magnitude difference in bytes scanned.

In [ ]:
# Scale the toy numbers to a realistic file size and a metered engine
# (Athena/BigQuery bill per TB scanned; $5/TB is the long-standing list price).
FILE_MB, PRICE_PER_TB, QUERIES_PER_DAY = 512, 5.0, 10_000
waste_tb = (files_all - files_one) * FILE_MB / 1_048_576

print(f"Hive user who forgets `WHERE dt=...`:  reads {files_all} files")
print(f"Iceberg user filtering on `ts`:        reads {files_one} files")
print(f"\nAt {FILE_MB} MB/file and ${PRICE_PER_TB:.0f}/TB scanned:")
print(f"  wasted per query: {waste_tb * 1024:.1f} GB  =  ${waste_tb * PRICE_PER_TB:.3f}")
print(f"  × {QUERIES_PER_DAY:,} queries/day  =  ${waste_tb * PRICE_PER_TB * QUERIES_PER_DAY:,.0f}/day")
print("\nThat is the bill for one forgotten predicate. Hidden partitioning removes")
print("the opportunity to forget.")

Hive user who forgets `WHERE dt=...`:  reads 10 files
Iceberg user filtering on `ts`:        reads 1 files

At 512 MB/file and $5/TB scanned:
  wasted per query: 4.5 GB  =  $0.022
  × 10,000 queries/day  =  $220/day

That is the bill for one forgotten predicate. Hidden partitioning removes
the opportunity to forget.


### 📝 Nhận xét — Cái giá của một predicate bị quên

- Người dùng Hive quên `WHERE dt=...` đọc **10/10 file**; người dùng Iceberg đọc **1 file**. Quy ra quy mô thật (512 MB/file, $5/TB scan như Athena/BigQuery): lãng phí (10 − 1) × 512 MB = **4,5 GB ≈ $0,022 mỗi query** → × 10.000 query/ngày = **≈ $220/ngày** (~$80.000/năm) chỉ vì một điều kiện lọc bị quên.
- Hidden partitioning không chỉ tối ưu hiệu năng — nó **loại bỏ khả năng mắc lỗi**: không có cột partition nào để quên.


## 5. The three-tier metadata tree

This is the structure the deck draws as a chain. Now walk it for real:

```
catalog  →  metadata.json  →  manifest list  →  manifest files  →  data files
            (schema, specs,   (one per         (file stats:
             snapshot ptr)     snapshot)        min/max, counts)
```

Every tier exists to let the tier above **skip** work below it.

In [ ]:
snaps = tbl.inspect.snapshots()
mans = tbl.inspect.manifests()
files = tbl.inspect.files()

print(f"Tier 1  metadata.json     : {tbl.metadata_location.rsplit('/', 1)[-1]}")
print(f"Tier 2  manifest lists    : {snaps.num_rows} (one per snapshot)")
print(f"Tier 3  manifest files    : {mans.num_rows}")
print(f"        data files        : {files.num_rows}")
print(f"\nMetadata log entries: {tbl.inspect.metadata_log_entries().num_rows}")
print(f"Partitions tracked  : {tbl.inspect.partitions().num_rows}")

Tier 1  metadata.json     : 00011-d2ddab14-0b08-45a7-9f82-6db237d1035a.metadata.json
Tier 2  manifest lists    : 10 (one per snapshot)
Tier 3  manifest files    : 10
        data files        : 10

Metadata log entries: 12
Partitions tracked  : 10


### 📝 Nhận xét — Cây metadata ba tầng

```
catalog → metadata.json → manifest list → manifest file → data file
          (schema, spec,   (1/snapshot)    (stats từng file:
           snapshot hiện tại)               partition, min/max, count)
```
- **Tier 1** `00011-….metadata.json`: số thứ tự 00011 ⇒ đã có 12 phiên bản metadata (create + đổi spec + 10 append) — khớp `Metadata log entries: 12`.
- **Tier 2** 10 manifest list — mỗi snapshot có một manifest list riêng.
- **Tier 3** snapshot hiện tại tham chiếu **10 manifest** (mỗi `append` thêm 1 manifest mới — *fast append*), mỗi manifest mô tả 1 data file; 10 partition được theo dõi.
- Mỗi tầng tồn tại để **tầng trên bỏ qua việc ở tầng dưới**: manifest list lưu khoảng giá trị partition của từng manifest ⇒ planner bỏ cả manifest không liên quan mà không mở; manifest lưu stats từng file ⇒ bỏ data file không cần đọc.


### The cost of planning

Metadata is not free. Count what a planner would have to open, and compare
metadata bytes to data bytes — this ratio is *why* Iceberg v4 is redesigning
the metadata tree, and why server-side planning matters at scale.

In [ ]:
loc = tbl.location().replace("file://", "")
meta_bytes = du(f"{loc}/metadata")
data_bytes = du(f"{loc}/data")
print(f"data/     {human(data_bytes):>10}   ({count_files(f'{loc}/data')} parquet files)")
print(f"metadata/ {human(meta_bytes):>10}   ({count_files(f'{loc}/metadata', '.avro')} avro + "
      f"{count_files(f'{loc}/metadata', '.json')} json)")
print(f"→ metadata is {meta_bytes / max(data_bytes, 1) * 100:.1f}% of table size")
print("\nAt 10 rows/file this looks absurd. At 512 MB/file it is ~0.1%.")
print("Small files punish you TWICE: more data files AND more metadata to plan over.")

data/        47.3 KB   (10 parquet files)
metadata/   139.1 KB   (20 avro + 12 json)
→ metadata is 294.2% of table size

At 10 rows/file this looks absurd. At 512 MB/file it is ~0.1%.
Small files punish you TWICE: more data files AND more metadata to plan over.


### 📝 Nhận xét — Metadata : data = 294%

- `data/` 47,3 KB (10 file Parquet) so với `metadata/` 139,1 KB (20 file Avro + 12 file JSON) ⇒ metadata = **294%** dung lượng dữ liệu.
- Vô lý ở đây vì mỗi file chỉ có 500 dòng (~4,7 KB); ở kích thước file thật (~512 MB) tỷ lệ này ~0,1%. Số file metadata tăng theo **số commit và số file**, không theo số dòng.
- Bài học giống NB2 nhưng nhìn từ phía metadata: **file nhỏ phạt hai lần** — nhiều data file phải mở *và* nhiều metadata phải đọc khi lập kế hoạch. Commit nhỏ, thường xuyên (streaming) làm cây metadata phình ra ⇒ cần compaction và snapshot expiry (NB6).


## 6. Schema evolution by field-ID (not by name, not by position)

Parquet columns are positional. Hive matched by name. Both break under
rename/reorder. Iceberg assigns every field a **permanent integer ID**;
names are just labels on top of it.

So a rename is a *metadata-only* operation — zero files rewritten.

In [ ]:
from pyiceberg.types import StringType  # noqa: E402

print("Field IDs before:", [(f.field_id, f.name) for f in tbl.schema().fields])

with tbl.update_schema() as upd:
    upd.add_column("tier", StringType(), doc="customer tier, added after 5000 rows existed")
tbl = cat.load_table(f"{ns}.llm_events")

with tbl.update_schema() as upd:
    upd.rename_column("latency_ms", "latency_millis")
tbl = cat.load_table(f"{ns}.llm_events")

print("Field IDs after :", [(f.field_id, f.name) for f in tbl.schema().fields])
print("\nlatency_ms → latency_millis kept field_id=4: a rename rewrote NO data.")
print("Old rows read back with tier=NULL — no backfill, no migration job.")

Field IDs before: [(1, 'event_id'), (2, 'ts'), (3, 'model'), (4, 'latency_ms'), (5, 'cost_usd')]


Field IDs after : [(1, 'event_id'), (2, 'ts'), (3, 'model'), (4, 'latency_millis'), (5, 'cost_usd'), (6, 'tier')]

latency_ms → latency_millis kept field_id=4: a rename rewrote NO data.
Old rows read back with tier=NULL — no backfill, no migration job.


In [ ]:
after = tbl.scan().to_arrow()
print(f"Rows still readable: {after.num_rows:,}")
print(f"Columns now: {after.column_names}")
print(f"tier nulls: {after.column('tier').null_count:,} (all pre-existing rows)")

Rows still readable: 5,000
Columns now: ['event_id', 'ts', 'model', 'latency_millis', 'cost_usd', 'tier']
tier nulls: 5,000 (all pre-existing rows)


### 📝 Nhận xét — Schema evolution theo field-ID

- Mỗi cột có **ID số nguyên vĩnh viễn**; tên chỉ là nhãn: `latency_ms` → `latency_millis` vẫn là **field-id 4**, cột mới `tier` nhận id **6** (id không bao giờ được tái sử dụng).
- Rename và add column là thao tác **chỉ sửa metadata — không rewrite file nào**. Toàn bộ **5.000 dòng vẫn đọc được**, `tier = null` cho cả 5.000 dòng cũ: không backfill, không migration job.
- Cơ chế: data file cũ **vẫn mang tên vật lý `latency_ms`** (kiểm chứng ở cell bằng chứng cuối §8) nhưng cột được gắn `PARQUET:field_id = 4`; reader ghép cột theo **ID**, không theo tên ⇒ đọc ra là `latency_millis`. File cũ không có field 6 ⇒ `tier` = null.
- So sánh: Hive ghép theo **tên** (rename ⇒ dữ liệu cũ đọc thành null), Parquet/CSV thuần ghép theo **vị trí** (reorder ⇒ lệch cột). Delta cũng hỗ trợ rename an toàn nhưng phải bật *column mapping*; với Iceberg field-ID là mặc định.


## 7. Time travel by snapshot

Same idea as Delta's `versionAsOf`, different spelling. Every snapshot is
addressable forever until you expire it (that's NB6).

In [ ]:
snap_ids = [s.snapshot_id for s in tbl.snapshots()]
first, last = snap_ids[0], snap_ids[-1]

print(f"snapshot[0]  {first}: {tbl.scan(snapshot_id=first).to_arrow().num_rows:,} rows")
print(f"snapshot[-1] {last}: {tbl.scan(snapshot_id=last).to_arrow().num_rows:,} rows")
print(f"\nTotal snapshots retained: {len(snap_ids)}")

hist = tbl.inspect.history().to_pylist()
print("\nHistory (last 3):")
for h in hist[-3:]:
    print(f"  {h['made_current_at']}  snapshot={h['snapshot_id']}")

snapshot[0]  4941317051457147094: 500 rows
snapshot[-1] 8221066054164963906: 5,000 rows

Total snapshots retained: 10

History (last 3):
  2026-10-05 04:48:59.576000  snapshot=2477717671976334822
  2026-10-05 04:48:59.630000  snapshot=8023879292769354568
  2026-10-05 04:48:59.701000  snapshot=8221066054164963906


### 📝 Nhận xét — Time travel theo snapshot

- `scan(snapshot_id=first)` → **500 dòng** (sau append đầu tiên); `snapshot_id=last` → **5.000 dòng**. Giống `versionAsOf` của Delta, nhưng định danh bằng snapshot ID thay vì số version liên tiếp.
- Vẫn **10 snapshot** dù §6 đã đổi schema hai lần: thay đổi schema/spec tạo **metadata.json mới nhưng không tạo snapshot**, vì dữ liệu không đổi.
- Mọi snapshot còn đọc được cho đến khi bị expire — chính sách giữ snapshot là job bảo trì của NB6.


## 8. Partition evolution — the thing Hive genuinely cannot do

Traffic grew; daily partitions are now too coarse. In Hive this is a
rewrite-the-whole-table migration. In Iceberg you change the spec and
**old data stays where it is** — each data file remembers which spec wrote it.

In [ ]:
from pyiceberg.transforms import IdentityTransform  # noqa: E402

with tbl.update_spec() as spec:
    spec.add_field("model", IdentityTransform(), "model_id")
tbl = cat.load_table(f"{ns}.llm_events")

tbl.append(day_batch(11).append_column("tier", pa.array(["gold"] * ROWS_PER_DAY))
           .rename_columns(["event_id", "ts", "model", "latency_millis", "cost_usd", "tier"]))
tbl = cat.load_table(f"{ns}.llm_events")

specs_in_use = set(tbl.inspect.files().column("spec_id").to_pylist())
print(f"Partition specs in use across data files: {sorted(specs_in_use)}")
print(f"Total rows readable across BOTH specs: {tbl.scan().to_arrow().num_rows:,}")
print("\nTwo layouts, one table, zero rewrites. This is the feature.")

Partition specs in use across data files: [1, 2]
Total rows readable across BOTH specs: 5,500

Two layouts, one table, zero rewrites. This is the feature.


In [ ]:
# Bằng chứng bổ sung: field-ID trong file vật lý, spec của từng file, pruning qua 2 layout
from collections import Counter
from pathlib import Path

import pyarrow.parquet as pq

fl = tbl.inspect.files().to_pylist()
print("Lịch sử spec  :", {s.spec_id: [f"{f.name}={f.transform}" for f in s.fields] for s in tbl.specs().values()})
print("Files / spec_id:", dict(sorted(Counter(x["spec_id"] for x in fl).items())))
for spec_id in (1, 2):
    x = next(x for x in fl if x["spec_id"] == spec_id)
    print(f"  ví dụ spec {spec_id}: {x['file_path'].split('llm_events/')[1][:62]}…  ({x['record_count']} rows)")

old = next(x for x in fl if x["spec_id"] == 1)
s = pq.read_schema(old["file_path"].replace("file://", ""))
print("\nFile CŨ (ghi trước rename) — cột vật lý + PARQUET:field_id:")
print("  ", [(n, s.field(n).metadata[b"PARQUET:field_id"].decode()) for n in s.names])
print("Schema bảng hiện tại:", [(f.name, f.field_id) for f in tbl.schema().fields])

print("\nPruning qua 2 layout:")
for flt in ["ts >= '2026-08-11T00:00:00' and model = 'claude-opus-4-7'", "model = 'claude-opus-4-7'"]:
    print(f"  {flt:58s} → {len(list(tbl.scan(row_filter=flt).plan_files()))}/{len(fl)} files")

n_meta = len(list(Path(tbl.location().replace("file://", ""), "metadata").glob("*.metadata.json")))
print(f"\nmetadata.json: {n_meta} file  |  snapshots: {len(tbl.snapshots())}  "
      f"(schema/spec change tạo metadata mới nhưng không tạo snapshot)")

### 📝 Nhận xét — Partition evolution

- Thêm `identity(model)` tạo **spec 2** (`ts_day` + `model_id`). Lịch sử spec: **0** (chưa partition) → **1** (`day(ts)`) → **2** (`day(ts)` + `model`).
- **10 file cũ giữ spec 1, không bị rewrite**; ngày 11 ghi theo spec 2 thành **3 file** (mỗi model một thư mục `ts_day=2026-08-11/model_id=…/`, ~167 dòng). Mỗi data file nhớ spec đã ghi ra nó ⇒ hai layout cùng tồn tại trong một bảng, **5.500 dòng đọc được** (spec đang dùng: `[1, 2]` ✅).
- Bằng chứng field-ID: file cũ (spec 1) có cột vật lý `('latency_ms', '4')` và không có `tier`, trong khi schema bảng là `('latency_millis', 4)`, `('tier', 6)` ⇒ rename thực sự không đụng vào dữ liệu.
- **Giới hạn quan trọng:** lọc `ts >= 08-11 AND model = 'opus'` chỉ đọc **1/13 file**, nhưng lọc riêng `model = 'opus'` phải đọc **11/13 file** — 10 file cũ không partition theo model nên không prune được. **Dữ liệu cũ không hưởng layout mới** cho đến khi được compact/rewrite.
- `metadata.json` tăng lên 16 file trong khi snapshot chỉ 11: rename, add column, đổi spec đều là commit **chỉ metadata**.
- Hive không làm được điều này: đổi partition ⇒ tạo bảng mới và rewrite toàn bộ dữ liệu.


## ✅ NB5 pass criteria

| Check | Target |
|---|---|
| Hidden-partition pruning ratio | ≥ 5× (printed in §4) |
| Snapshots retained | ≥ 10 |
| Schema evolved, field IDs stable | `latency_millis` keeps `field_id=4` |
| Partition evolution | ≥ 2 distinct `spec_id`s, table still fully readable |

In [ ]:
checks = {
    "pruning ratio ≥ 5x":        PRUNE_RATIO >= 5,
    "≥ 10 snapshots":            len(tbl.snapshots()) >= 10,
    "field_id stable on rename": [f.field_id for f in tbl.schema().fields if f.name == "latency_millis"] == [4],
    "≥ 2 partition specs":       len(specs_in_use) >= 2,
    "all rows readable":         tbl.scan().to_arrow().num_rows == (N_DAYS + 1) * ROWS_PER_DAY,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB5 incomplete — see FAIL rows above"
print("\nNB5 complete.")

  [PASS] pruning ratio ≥ 5x
  [PASS] ≥ 10 snapshots
  [PASS] field_id stable on rename
  [PASS] ≥ 2 partition specs
  [PASS] all rows readable

NB5 complete.


### 🎯 Kết luận NB5

Iceberg đạt cùng mục tiêu ACID/time travel như Delta nhưng đặt **catalog** làm trung tâm: catalog sở hữu vị trí bảng và con trỏ metadata, mọi commit là một lần đổi con trỏ. **Hidden partitioning** lưu transform `day(ts)` trong metadata nên lọc trên `ts` cho pruning **10×** mà không thể "quên predicate" như Hive (~$220/ngày ở quy mô thật). Cây metadata ba tầng cho phép bỏ qua công việc ở từng tầng, nhưng chi phí metadata tăng theo số file/commit (294% ở bảng file nhỏ này). **Field-ID** làm rename/add column thành thao tác chỉ-metadata (file cũ vẫn mang tên `latency_ms`), và **partition evolution** cho hai spec cùng tồn tại không cần rewrite — với giới hạn là dữ liệu cũ không tự hưởng layout mới.
